# 🎯 SIRCCD — Anonimización: Dataset + Entrenamiento YOLO11s

Notebook completo que descarga los datasets, los convierte a formato YOLO, y entrena un modelo **YOLO11s Detect** para detectar **rostros** y **placas vehiculares**.

> ✅ **Todo automático** — descarga, conversión, entrenamiento y evaluación en un solo notebook.

### Datasets
| Dataset | Clase | Imágenes | Descarga |
|---------|-------|----------|----------|
| **WIDER FACE** | `face` | ~32,000 | gdown (Google Drive) |
| **CCPD** | `license_plate` | ~250,000 (→ 15,000) | gdown (Google Drive) |
| **RodoSol-ALPR** | `license_plate` | ~20,000 (→ 5,000) | GitHub (zip) |

### Modelo
| Propiedad | Valor |
|-----------|-------|
| Base | YOLO11s (Ultralytics, pre-entrenado COCO) |
| Clases | `face` (0), `license_plate` (1) |
| Input | 640×640 |
| Épocas | 150 |
| Batch | 64 (A100 — 40 GB VRAM) |

### Métricas objetivo
- mAP@0.5 ≥ 0.82
- Recall ≥ 0.87 (priorizamos NO omitir rostros/placas)

### Requisitos
- [ ] GPU **A100** (Runtime → Change runtime type → A100)
- [ ] Espacio disponible: ~30 GB en disco de Colab

## 🔧 1. Setup

In [ ]:
# Instalar dependencias
!pip install -q ultralytics>=8.3.0 gdown Pillow tqdm

import torch
import os
import shutil
import random
import json
import zipfile
from pathlib import Path
from collections import Counter

# Verificar GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'✅ GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)')
else:
    print('❌ GPU no disponible — ve a Runtime → Change runtime type → A100')
    raise SystemError('GPU requerida')

# Batch size optimizado para A100
if vram_gb >= 40:
    BATCH = 64
elif vram_gb >= 16:
    BATCH = 32
else:
    BATCH = 16

if 'A100' not in gpu_name:
    print(f'⚠️ GPU: {gpu_name} — Se recomienda A100 para máximo rendimiento')

print(f'📦 Batch size: {BATCH}')

# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Directorios
WORK_DIR = '/content/anonymization'
DRIVE_DIR = '/content/drive/MyDrive/sirccd/anonymization'
RAW_DIR = f'{WORK_DIR}/raw'
PROCESSED_DIR = f'{WORK_DIR}/processed'
DATASET_DIR = f'{WORK_DIR}/dataset'
SEED = 42

for d in [RAW_DIR, PROCESSED_DIR, DATASET_DIR, DRIVE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'\n✅ Directorio de trabajo: {WORK_DIR}')
print(f'✅ Directorio en Drive: {DRIVE_DIR}')

---
# 📥 PARTE 1: Descarga y Preparación del Dataset
---

## 📥 2. Descargar WIDER FACE (Rostros)

**393,703 bounding boxes** en 32,203 imágenes. Fuente: http://shuoyang1213.me/WIDERFACE/

In [ ]:
import gdown

WIDER_DIR = f'{RAW_DIR}/wider_face'
os.makedirs(WIDER_DIR, exist_ok=True)

wider_files = {
    'WIDER_train.zip': '15hGDLhsx8bLgLcIRD5DhYt5iBxnjNF1M',
    'WIDER_val.zip':   '1GUCogbp16PMGa39thoMMeWxp7Rp5oM8Q',
    'wider_face_split.zip': '1H2n4TTFG7Q-x2W5CWE1M8JbMnW3KxJk5',
}

for filename, file_id in wider_files.items():
    dest = f'{WIDER_DIR}/{filename}'
    if os.path.exists(dest.replace('.zip', '')):
        print(f'✅ {filename} ya descargado')
        continue
    print(f'\n⬇️  Descargando {filename}...')
    gdown.download(id=file_id, output=dest, quiet=False)
    print(f'📦 Extrayendo {filename}...')
    with zipfile.ZipFile(dest, 'r') as z:
        z.extractall(WIDER_DIR)
    os.remove(dest)

for d in ['WIDER_train/images', 'WIDER_val/images', 'wider_face_split']:
    path = f'{WIDER_DIR}/{d}'
    if os.path.exists(path):
        count = sum(1 for _ in os.scandir(path))
        print(f'  ✅ {d}: {count} items')
    else:
        print(f'  ❌ {d}: NO ENCONTRADO')

## 📥 3. Descargar CCPD (Placas — China)

**CCPD** — ~250,000 imágenes de placas chinas. Coordenadas en el nombre del archivo. Submuestreo a **15,000**.

Fuente: https://github.com/detectRecog/CCPD

In [ ]:
CCPD_DIR = f'{RAW_DIR}/ccpd'
os.makedirs(CCPD_DIR, exist_ok=True)

CCPD_ZIP = f'{CCPD_DIR}/ccpd_base.zip'
CCPD_BASE = f'{CCPD_DIR}/ccpd_base'

if os.path.exists(CCPD_BASE) and len(os.listdir(CCPD_BASE)) > 100:
    count = len(os.listdir(CCPD_BASE))
    print(f'✅ CCPD ya descargado: {count} imágenes')
else:
    CCPD_GDRIVE_ID = '1fFqCXjhk7vTijbUX-wXELfNJuOFMz5qd'
    print('⬇️  Descargando CCPD base desde Google Drive...')
    gdown.download(id=CCPD_GDRIVE_ID, output=CCPD_ZIP, quiet=False)

    if os.path.exists(CCPD_ZIP):
        print('📦 Extrayendo...')
        with zipfile.ZipFile(CCPD_ZIP, 'r') as z:
            z.extractall(CCPD_DIR)
        os.remove(CCPD_ZIP)

        if not os.path.exists(CCPD_BASE):
            for d in os.listdir(CCPD_DIR):
                full = os.path.join(CCPD_DIR, d)
                if os.path.isdir(full) and d != 'ccpd_base':
                    imgs = [f for f in os.listdir(full) if f.endswith(('.jpg', '.png'))]
                    if len(imgs) > 100:
                        os.rename(full, CCPD_BASE)
                        break

        if os.path.exists(CCPD_BASE):
            print(f'✅ CCPD extraído: {len(os.listdir(CCPD_BASE))} imágenes')
        else:
            print('⚠️ Carpeta ccpd_base no encontrada. Verifica manualmente.')
    else:
        print('❌ Descarga fallida. Ve a: https://github.com/detectRecog/CCPD')

## 📥 4. Descargar RodoSol-ALPR (Placas — Brasil)

**20,000 imágenes** de peajes brasileños. Submuestreo a **5,000**. Descarga directa desde GitHub.

Fuente: https://github.com/raysonlaroca/rodosol-alpr-dataset

In [ ]:
RODOSOL_DIR = f'{RAW_DIR}/rodosol_alpr'
os.makedirs(RODOSOL_DIR, exist_ok=True)

RODOSOL_ZIP = f'{RODOSOL_DIR}/rodosol.zip'
existing_imgs = sum(1 for r, _, files in os.walk(RODOSOL_DIR) for f in files if f.endswith(('.png', '.jpg')))

if existing_imgs > 100:
    print(f'✅ RodoSol-ALPR ya descargado: {existing_imgs} imágenes')
else:
    RODOSOL_URL = 'https://github.com/raysonlaroca/rodosol-alpr-dataset/archive/refs/heads/master.zip'
    print('⬇️  Descargando RodoSol-ALPR desde GitHub...')
    !wget -q --show-progress -O {RODOSOL_ZIP} {RODOSOL_URL}

    if os.path.exists(RODOSOL_ZIP) and os.path.getsize(RODOSOL_ZIP) > 1000:
        print('📦 Extrayendo...')
        with zipfile.ZipFile(RODOSOL_ZIP, 'r') as z:
            z.extractall(RODOSOL_DIR)
        os.remove(RODOSOL_ZIP)

        for root, dirs, files in os.walk(RODOSOL_DIR):
            imgs = [f for f in files if f.endswith(('.png', '.jpg'))]
            if len(imgs) > 100:
                print(f'  📁 Imágenes encontradas en: {root}')
                break

        total = sum(1 for r, _, files in os.walk(RODOSOL_DIR) for f in files if f.endswith(('.png', '.jpg')))
        print(f'✅ RodoSol-ALPR extraído: {total} imágenes')
    else:
        print('❌ Descarga fallida. Ve a: https://github.com/raysonlaroca/rodosol-alpr-dataset')

## 🔄 5. Convertir WIDER FACE → Formato YOLO

Convierte anotaciones de WIDER FACE a formato YOLO normalizado: `clase cx cy w h` (clase `0` = `face`)

In [ ]:
from PIL import Image
from tqdm import tqdm

FACE_CLASS = 0
WIDER_YOLO = f'{PROCESSED_DIR}/wider_face_yolo'


def convert_wider_split(ann_path, images_root, split):
    out_imgs = Path(f'{WIDER_YOLO}/images/{split}')
    out_lbls = Path(f'{WIDER_YOLO}/labels/{split}')
    out_imgs.mkdir(parents=True, exist_ok=True)
    out_lbls.mkdir(parents=True, exist_ok=True)

    with open(ann_path, 'r') as f:
        lines = f.readlines()

    idx = 0
    total = 0
    pbar = tqdm(desc=f'[{split}] Convirtiendo', unit='img')

    while idx < len(lines):
        img_rel = lines[idx].strip()
        idx += 1
        img_path = Path(images_root) / img_rel

        if not img_path.exists():
            num = int(lines[idx].strip())
            idx += 1
            idx += max(num, 1)
            continue

        num_faces = int(lines[idx].strip())
        idx += 1

        if num_faces == 0:
            idx += 1
            continue

        with Image.open(img_path) as img:
            img_w, img_h = img.size

        yolo_lines = []
        for _ in range(num_faces):
            parts = lines[idx].strip().split()
            idx += 1
            x1, y1, w, h = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])

            if w <= 0 or h <= 0:
                continue
            if len(parts) >= 8 and int(parts[7]) == 1:
                continue

            cx = max(0, min(1, (x1 + w / 2) / img_w))
            cy = max(0, min(1, (y1 + h / 2) / img_h))
            nw = max(0.001, min(1, w / img_w))
            nh = max(0.001, min(1, h / img_h))
            yolo_lines.append(f'{FACE_CLASS} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')

        if not yolo_lines:
            continue

        safe_name = img_rel.replace('/', '_').replace('\\', '_')
        stem = Path(safe_name).stem
        suffix = img_path.suffix

        shutil.copy2(img_path, out_imgs / f'{stem}{suffix}')
        with open(out_lbls / f'{stem}.txt', 'w') as lf:
            lf.write('\n'.join(yolo_lines) + '\n')

        total += 1
        pbar.update(1)

    pbar.close()
    return total


train_out = Path(f'{WIDER_YOLO}/images/train')
if train_out.exists() and sum(1 for _ in train_out.iterdir()) > 100:
    print(f'✅ WIDER FACE ya convertido: {sum(1 for _ in train_out.iterdir())} imgs train')
else:
    ann_dir = f'{WIDER_DIR}/wider_face_split'

    print('🔄 Convirtiendo WIDER FACE train...')
    n_train = convert_wider_split(
        f'{ann_dir}/wider_face_train_bbx_gt.txt',
        f'{WIDER_DIR}/WIDER_train/images', 'train'
    )
    print(f'  → {n_train} imágenes')

    print('🔄 Convirtiendo WIDER FACE val...')
    n_val = convert_wider_split(
        f'{ann_dir}/wider_face_val_bbx_gt.txt',
        f'{WIDER_DIR}/WIDER_val/images', 'val'
    )
    print(f'  → {n_val} imágenes')
    print(f'\n✅ WIDER FACE convertido — Total: {n_train + n_val}')

## 🔄 6. Convertir Placas → Formato YOLO

Convierte **CCPD** (submuestreo 15 000) y **RodoSol-ALPR** (submuestreo 5 000) a formato YOLO.

In [ ]:
import random
from PIL import Image
from tqdm import tqdm

PLATE_CLASS = 1
PLATES_YOLO = f'{PROCESSED_DIR}/plates_yolo'
CCPD_SUBSAMPLE = 15_000
RODOSOL_SUBSAMPLE = 5_000

# ─── CCPD ───────────────────────────────────────────────────────
def convert_ccpd(ccpd_root, max_images=CCPD_SUBSAMPLE):
    out_imgs = Path(f'{PLATES_YOLO}/images/ccpd')
    out_lbls = Path(f'{PLATES_YOLO}/labels/ccpd')
    out_imgs.mkdir(parents=True, exist_ok=True)
    out_lbls.mkdir(parents=True, exist_ok=True)

    all_jpgs = sorted(Path(ccpd_root).rglob('*.jpg'))
    random.seed(SEED)
    if len(all_jpgs) > max_images:
        all_jpgs = random.sample(all_jpgs, max_images)
    print(f'  CCPD: procesando {len(all_jpgs)} imágenes')

    ok = 0
    for img_path in tqdm(all_jpgs, desc='CCPD', unit='img'):
        parts = img_path.stem.split('-')
        if len(parts) < 4:
            continue
        try:
            tl = parts[2].split('_')
            br = parts[3].split('_')
            tl = [int(tl[0].replace('&', '')), int(tl[1].replace('&', ''))]
            br = [int(br[0].replace('&', '')), int(br[1].replace('&', ''))]
        except (ValueError, IndexError):
            continue

        with Image.open(img_path) as img:
            w, h = img.size

        cx = max(0, min(1, ((tl[0] + br[0]) / 2) / w))
        cy = max(0, min(1, ((tl[1] + br[1]) / 2) / h))
        nw = max(0.001, min(1, abs(br[0] - tl[0]) / w))
        nh = max(0.001, min(1, abs(br[1] - tl[1]) / h))

        shutil.copy2(img_path, out_imgs / img_path.name)
        with open(out_lbls / f'{img_path.stem}.txt', 'w') as lf:
            lf.write(f'{PLATE_CLASS} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n')
        ok += 1
    return ok

# ─── RodoSol ────────────────────────────────────────────────────
def convert_rodosol(rodosol_root, max_images=RODOSOL_SUBSAMPLE):
    out_imgs = Path(f'{PLATES_YOLO}/images/rodosol')
    out_lbls = Path(f'{PLATES_YOLO}/labels/rodosol')
    out_imgs.mkdir(parents=True, exist_ok=True)
    out_lbls.mkdir(parents=True, exist_ok=True)

    ann_dir = Path(rodosol_root) / 'annotations'
    img_dir = Path(rodosol_root) / 'images'

    if not ann_dir.exists():
        for candidate in Path(rodosol_root).rglob('annotations'):
            if candidate.is_dir():
                ann_dir = candidate
                img_dir = candidate.parent / 'images'
                break

    all_txts = sorted(ann_dir.glob('*.txt'))
    random.seed(SEED)
    if len(all_txts) > max_images:
        all_txts = random.sample(all_txts, max_images)
    print(f'  RodoSol: procesando {len(all_txts)} anotaciones')

    ok = 0
    for ann_path in tqdm(all_txts, desc='RodoSol', unit='img'):
        data = {}
        with open(ann_path, 'r') as f:
            for line in f:
                line = line.strip()
                if ':' in line:
                    key, val = line.split(':', 1)
                    data[key.strip()] = val.strip()

        if 'position_plate' not in data:
            continue

        parts = data['position_plate'].split()
        if len(parts) < 4:
            continue

        x, y, w_box, h_box = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])

        img_path = None
        for ext in ['.jpg', '.png', '.jpeg']:
            candidate = img_dir / f'{ann_path.stem}{ext}'
            if candidate.exists():
                img_path = candidate
                break

        if img_path is None:
            continue

        with Image.open(img_path) as img:
            img_w, img_h = img.size

        cx = max(0, min(1, (x + w_box / 2) / img_w))
        cy = max(0, min(1, (y + h_box / 2) / img_h))
        nw = max(0.001, min(1, w_box / img_w))
        nh = max(0.001, min(1, h_box / img_h))

        shutil.copy2(img_path, out_imgs / img_path.name)
        with open(out_lbls / f'{ann_path.stem}.txt', 'w') as lf:
            lf.write(f'{PLATE_CLASS} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n')
        ok += 1
    return ok


# --- Ejecutar conversiones ---
ccpd_out = Path(f'{PLATES_YOLO}/images/ccpd')
if ccpd_out.exists() and sum(1 for _ in ccpd_out.iterdir()) > 100:
    print(f'✅ Placas ya convertidas — CCPD: {sum(1 for _ in ccpd_out.iterdir())}')
else:
    print('🔄 Convirtiendo CCPD...')
    ccpd_dirs = sorted(Path(CCPD_DIR).glob('ccpd_*'))
    if not ccpd_dirs:
        ccpd_dirs = [Path(CCPD_DIR)]
    n_ccpd = convert_ccpd(ccpd_dirs[0] if len(ccpd_dirs) == 1 else CCPD_DIR)
    print(f'  → {n_ccpd} imágenes')

    print('🔄 Convirtiendo RodoSol...')
    rodosol_root = RODOSOL_DIR
    for candidate in Path(RODOSOL_DIR).iterdir():
        if candidate.is_dir() and (candidate / 'images').exists():
            rodosol_root = str(candidate)
            break
    n_rodo = convert_rodosol(rodosol_root)
    print(f'  → {n_rodo} imágenes')
    print(f'\n✅ Placas convertidas — CCPD: {n_ccpd}, RodoSol: {n_rodo}')

## 🔀 7. Unificar y Dividir Dataset

Combina **rostros** (WIDER FACE) + **placas** (CCPD + RodoSol) y divide en:

| Split  | Proporción |
|--------|-----------|
| Train  | 85 %      |
| Val    | 10 %      |
| Test   | 5 %       |

In [ ]:
import json
from collections import Counter

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.85, 0.10, 0.05

def collect_pairs(base_dir):
    """Recopila pares (imagen, label) de un directorio YOLO."""
    pairs = []
    base = Path(base_dir)

    for split_or_src in ['train', 'val', 'ccpd', 'rodosol']:
        img_dir = base / 'images' / split_or_src
        lbl_dir = base / 'labels' / split_or_src
        if not img_dir.exists():
            continue
        for img_path in sorted(img_dir.iterdir()):
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                lbl_path = lbl_dir / f'{img_path.stem}.txt'
                if lbl_path.exists():
                    pairs.append((img_path, lbl_path))
    return pairs


# Recoger todas las parejas
all_pairs = []
all_pairs.extend(collect_pairs(WIDER_YOLO))
all_pairs.extend(collect_pairs(PLATES_YOLO))

print(f'📊 Total de pares imagen+label: {len(all_pairs)}')

# Shuffle determinístico
random.seed(SEED)
random.shuffle(all_pairs)

# Dividir
n = len(all_pairs)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

splits = {
    'train': all_pairs[:n_train],
    'val': all_pairs[n_train:n_train + n_val],
    'test': all_pairs[n_train + n_val:]
}

# Copiar a directorio final
for split_name, pairs in splits.items():
    img_out = Path(f'{DATASET_DIR}/images/{split_name}')
    lbl_out = Path(f'{DATASET_DIR}/labels/{split_name}')
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_path, lbl_path in tqdm(pairs, desc=f'Copiando {split_name}', unit='file'):
        # Evitar colisiones de nombres
        dest_name = img_path.name
        counter = 1
        while (img_out / dest_name).exists():
            dest_name = f'{img_path.stem}_{counter}{img_path.suffix}'
            counter += 1

        shutil.copy2(img_path, img_out / dest_name)
        lbl_dest = dest_name.rsplit('.', 1)[0] + '.txt'
        shutil.copy2(lbl_path, lbl_out / lbl_dest)

# Estadísticas por clase
class_names = {0: 'face', 1: 'license_plate'}
report = {}

for split_name in ['train', 'val', 'test']:
    lbl_dir = Path(f'{DATASET_DIR}/labels/{split_name}')
    counts = Counter()
    n_imgs = sum(1 for _ in Path(f'{DATASET_DIR}/images/{split_name}').iterdir())

    for lbl_file in lbl_dir.glob('*.txt'):
        with open(lbl_file) as f:
            for line in f:
                cls_id = int(line.strip().split()[0])
                counts[cls_id] += 1

    report[split_name] = {'images': n_imgs, 'annotations': dict(counts)}
    print(f'\n  {split_name}: {n_imgs} imágenes')
    for cls_id, count in sorted(counts.items()):
        print(f'    clase {cls_id} ({class_names.get(cls_id, "?")}): {count} anotaciones')

# Guardar reporte
report_path = f'{DATASET_DIR}/dataset_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'\n✅ Dataset unificado en {DATASET_DIR}')
print(f'   Reporte guardado en {report_path}')

## ✅ 8. Validación del Dataset

Verifica la integridad: cada imagen debe tener su label y viceversa.

In [ ]:
all_ok = True

for split in ['train', 'val', 'test']:
    img_dir = Path(f'{DATASET_DIR}/images/{split}')
    lbl_dir = Path(f'{DATASET_DIR}/labels/{split}')

    imgs = {p.stem for p in img_dir.iterdir() if p.suffix.lower() in ['.jpg', '.jpeg', '.png']}
    lbls = {p.stem for p in lbl_dir.glob('*.txt')}

    missing_labels = imgs - lbls
    missing_images = lbls - imgs

    if missing_labels or missing_images:
        all_ok = False
        print(f'⚠️  {split}:')
        if missing_labels:
            print(f'   Imágenes sin label: {len(missing_labels)}')
        if missing_images:
            print(f'   Labels sin imagen: {len(missing_images)}')
    else:
        print(f'✅ {split}: {len(imgs)} pares OK')

if all_ok:
    print('\n🎉 Dataset válido — listo para entrenamiento')
else:
    print('\n⚠️  Hay inconsistencias — revisar antes de entrenar')

## 💾 9. Respaldar Dataset en Drive

Crea un ZIP del dataset unificado y lo guarda en Google Drive para reutilizar sin reprocesar.

In [ ]:
backup_zip = f'{DRIVE_BASE}/sirccd_anon_dataset.zip'

if Path(backup_zip).exists():
    print(f'✅ Backup ya existe: {backup_zip}')
    size_mb = Path(backup_zip).stat().st_size / (1024 * 1024)
    print(f'   Tamaño: {size_mb:.1f} MB')
else:
    print('📦 Creando ZIP del dataset...')
    import subprocess
    result = subprocess.run(
        ['zip', '-r', '-q', backup_zip, '.'],
        cwd=DATASET_DIR,
        capture_output=True, text=True
    )
    if result.returncode == 0:
        size_mb = Path(backup_zip).stat().st_size / (1024 * 1024)
        print(f'✅ Backup guardado: {backup_zip} ({size_mb:.1f} MB)')
    else:
        print(f'❌ Error creando ZIP: {result.stderr}')

---

# 🏋️ PARTE 2: Entrenamiento

A partir de aquí se entrena el modelo YOLO11s con el dataset unificado.

## 📝 10. Configurar `data.yaml`

In [ ]:
import yaml

data_yaml_path = f'{WORK_DIR}/data.yaml'

# Regenerar con paths absolutos de Colab
data_config = {
    'path': DATASET_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 2,
    'names': {0: 'face', 1: 'license_plate'},
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print('✅ data.yaml:')
!cat {data_yaml_path}

## 🏋️ 11. Entrenar YOLO11s

| Parámetro | Valor |
|-----------|-------|
| Épocas | 150 |
| Batch | 64 (A100) |
| Imagen | 640×640 |
| Optimizador | AdamW |
| Paciencia | 30 |

In [ ]:
from ultralytics import YOLO

# Cargar modelo base pre-entrenado en COCO
model = YOLO('yolo11s.pt')

PROJECT = f'{WORK_DIR}/runs'
NAME = 'anonymizer_v1'

results = model.train(
    data=data_yaml_path,
    epochs=150,
    imgsz=640,
    batch=BATCH,
    project=PROJECT,
    name=NAME,
    exist_ok=True,

    # Optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=5,
    warmup_momentum=0.8,

    # Early stopping
    patience=30,

    # Augmentations
    flipud=0.0,       # NO invertir verticalmente (rostros)
    fliplr=0.5,       # Sí espejo horizontal
    degrees=5.0,      # Ligera rotación
    scale=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,

    # Misc
    workers=2,
    val=True,
    plots=True,
    save=True,
    save_period=25,  # Checkpoint cada 25 epochs
    device=0,
    verbose=True,
)

print(f'\n✅ Entrenamiento completado')
print(f'  Mejor modelo: {PROJECT}/{NAME}/weights/best.pt')

## 📈 12. Curvas de Entrenamiento

In [ ]:
from IPython.display import Image, display

run_dir = f'{PROJECT}/{NAME}'

for plot in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
             'PR_curve.png', 'F1_curve.png']:
    plot_path = f'{run_dir}/{plot}'
    if os.path.exists(plot_path):
        print(f'\n--- {plot} ---')
        display(Image(filename=plot_path, width=800))
    else:
        print(f'  {plot}: no encontrado')

## 🧪 13. Evaluación en Test Set

In [ ]:
# Cargar mejor modelo
best_model = YOLO(f'{run_dir}/weights/best.pt')

# Evaluar en test
test_results = best_model.val(
    data=data_yaml_path,
    split='test',
    imgsz=640,
    batch=BATCH,
    device=0,
    verbose=True,
)

# Mostrar métricas
print('\n' + '='*60)
print('🎯 MÉTRICAS EN TEST SET')
print('='*60)

map50 = test_results.results_dict.get('metrics/mAP50(B)', 0)
map50_95 = test_results.results_dict.get('metrics/mAP50-95(B)', 0)
precision = test_results.results_dict.get('metrics/precision(B)', 0)
recall = test_results.results_dict.get('metrics/recall(B)', 0)

print(f'  mAP@0.5:     {map50:.4f}  {"\u2705" if map50 >= 0.82 else "\u26a0\ufe0f <0.82"}')
print(f'  mAP@0.5:0.95:{map50_95:.4f}')
print(f'  Precision:   {precision:.4f}')
print(f'  Recall:      {recall:.4f}  {"\u2705" if recall >= 0.87 else "\u26a0\ufe0f <0.87"}')

# Per-class
class_names = ['face', 'license_plate']
print(f'\n📊 Por clase:')
if hasattr(test_results, 'box') and hasattr(test_results.box, 'maps'):
    for i, name in enumerate(class_names):
        if i < len(test_results.box.maps):
            print(f'  {name}: mAP@0.5:0.95 = {test_results.box.maps[i]:.4f}')

## 👀 14. Visualizar Predicciones

In [ ]:
import matplotlib.pyplot as plt
import random
from PIL import Image as PILImage
import numpy as np

test_imgs = sorted(Path(f'{DATASET_DIR}/images/test').glob('*'))[:50]
random.seed(42)
random.shuffle(test_imgs)
samples = test_imgs[:9]

fig, axes = plt.subplots(3, 3, figsize=(18, 18))
fig.suptitle('Predicciones en Test Set', fontsize=16)

for ax, img_path in zip(axes.flat, samples):
    preds = best_model.predict(str(img_path), conf=0.25, iou=0.45, verbose=False)
    result_img = preds[0].plot()  # BGR
    result_rgb = result_img[:, :, ::-1]  # → RGB
    ax.imshow(result_rgb)
    ax.set_title(img_path.name[:35], fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 🔍 15. Test de Anonimización (Detección + Blur)

In [ ]:
import cv2

MARGIN = 0.20

def anonymize_image(model, img_path, conf=0.25, iou=0.45):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    preds = model.predict(str(img_path), conf=conf, iou=iou, verbose=False)

    for box in preds[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        bw, bh = x2 - x1, y2 - y1
        margin_x = int(bw * MARGIN)
        margin_y = int(bh * MARGIN)

        x1 = max(0, x1 - margin_x)
        y1 = max(0, y1 - margin_y)
        x2 = min(w, x2 + margin_x)
        y2 = min(h, y2 + margin_y)

        roi = img[y1:y2, x1:x2]
        blurred = cv2.GaussianBlur(roi, (51, 51), 30)
        img[y1:y2, x1:x2] = blurred

    return img, len(preds[0].boxes)


# Mostrar comparación: original vs anonimizado
fig, axes = plt.subplots(3, 2, figsize=(16, 20))
fig.suptitle('Original vs Anonimizado', fontsize=16)

for i, img_path in enumerate(samples[:3]):
    orig = cv2.imread(str(img_path))
    anon, n_det = anonymize_image(best_model, img_path)

    axes[i][0].imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB))
    axes[i][0].set_title(f'Original: {img_path.name[:30]}')
    axes[i][0].axis('off')

    axes[i][1].imshow(cv2.cvtColor(anon, cv2.COLOR_BGR2RGB))
    axes[i][1].set_title(f'Anonimizado ({n_det} detecciones)')
    axes[i][1].axis('off')

plt.tight_layout()
plt.show()

## 💾 16. Guardar Modelo en Drive

In [ ]:
import shutil

MODELS_DRIVE = f'{DRIVE_DIR}/models'
os.makedirs(MODELS_DRIVE, exist_ok=True)

best_pt = f'{run_dir}/weights/best.pt'
last_pt = f'{run_dir}/weights/last.pt'

# Copiar pesos
for src in [best_pt, last_pt]:
    if os.path.exists(src):
        dst = f'{MODELS_DRIVE}/{Path(src).name}'
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / (1024 * 1024)
        print(f'✅ {Path(src).name} → Drive ({size_mb:.1f} MB)')

# Copiar resultados
for f in ['results.csv', 'results.png', 'confusion_matrix.png',
          'PR_curve.png', 'F1_curve.png', 'args.yaml']:
    src = f'{run_dir}/{f}'
    if os.path.exists(src):
        shutil.copy2(src, f'{MODELS_DRIVE}/{f}')

print(f'\n✅ Modelo y métricas guardados en: {MODELS_DRIVE}')
print(f'\n💡 Para usar en producción, copia best.pt a:')
print(f'   backend/models/anonymizer/best.pt')
print(f'   ml/anonymization/models/best.pt')

## 🔁 17. Reanudar Entrenamiento (Opcional)

Si se desconecta Colab, puedes reanudar desde el último checkpoint.

In [ ]:
# ⚠️ Solo ejecutar si necesitas reanudar
# resume_model = YOLO(f'{run_dir}/weights/last.pt')
# resume_model.train(resume=True)

## 📤 18. Exportar Modelo (Opcional)

Exportar a ONNX para inferencia optimizada en producción.

In [ ]:
# Exportar a ONNX
export_model = YOLO(f'{run_dir}/weights/best.pt')
export_path = export_model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    dynamic=False,
)
print(f'\n✅ Modelo exportado: {export_path}')

# Copiar a Drive
if os.path.exists(export_path):
    shutil.copy2(export_path, f'{MODELS_DRIVE}/best.onnx')
    print(f'✅ ONNX copiado a Drive')

---
## ✅ Resumen

### Archivos guardados en Drive
```
Drive/sirccd/anonymization/
├── sirccd_anon_dataset.zip     ← Backup del dataset
└── models/
    ├── best.pt                 ← Mejor modelo (para backend)
    ├── last.pt                 ← Último checkpoint
    ├── best.onnx               ← Exportación ONNX (opcional)
    ├── results.csv             ← Métricas por época
    ├── results.png             ← Curvas de entrenamiento
    ├── confusion_matrix.png
    ├── PR_curve.png
    └── args.yaml               ← Hiperparámetros usados
```

### Siguiente paso
Copia `best.pt` al backend del proyecto:
```bash
cp Drive/sirccd/anonymization/models/best.pt  backend/models/anonymizer/best.pt
```